## Gene analysis for fig 1

In [1]:
import pandas as pd
import glob
import re

data_dir = '/private/groups/migalab/juklucas/censat_regions/censat_arrays/tables/censat_regions_pass_qc_by_chrom'

dfs = []
for f in glob.glob(f'{data_dir}/censat_regions_pass_qc_chr*.tsv'):
    chrom = re.search(r'chr[\dXYM]+', f).group(0)
    df = pd.read_csv(f, sep='\t')
    df['chrom'] = chrom
    dfs.append(df)

censat_df = pd.concat(dfs, ignore_index=True)
print(censat_df.shape)
censat_df.head()


(4147, 26)


,sample_id,haplotype,assembly_id,sequence_id,region_start,region_end,region_size_bp,feature_count,region_label_bp_json,simplified_label_bp_json,...,contig_size,dist_to_contig_start,dist_to_contig_end,region_not_near_contig_edge,reference_large_array_labels_json,all_large_arrays_represented,sequences_per_assembly_chrom,single_sequence_for_chrom,pass_qc,chrom
0,CHM13,0,chm13v2.0_maskedY_rCRS,CHM13#0#chr18,14298107,21125235,6827128,33,"{""active_hor(S2C18H1L)"": 4967510, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6819, ""active"": 496751...",...,80542538.0,14298107,59417303.0,True,"[""active""]",True,1.0,True,True,chr18
1,HG00097,1,HG00097_hap1_hprc_r2_v1.0.1,HG00097#1#JBIRDD010000010.1,14135694,19270914,5135220,40,"{""active_hor(S2C18H1L)"": 3113600, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 311360...",...,78710895.0,14135694,59439981.0,True,"[""active""]",True,1.0,True,True,chr18
2,HG00097,2,HG00097_hap2_hprc_r2_v1.0.1,HG00097#2#CM094084.1,14138081,20853464,6715383,33,"{""active_hor(S2C18H1L)"": 4697016, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 469701...",...,80317024.0,14138081,59463560.0,True,"[""active""]",True,1.0,True,True,chr18
3,HG00099,1,HG00099_hap1_hprc_r2_v1.0.1,HG00099#1#CM087325.1,14150536,20957234,6806698,37,"{""active_hor(S2C18H1L)"": 4765242, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 476524...",...,80476601.0,14150536,59519367.0,True,"[""active""]",True,1.0,True,True,chr18
4,HG00099,2,HG00099_hap2_hprc_r2_v1.0.1,HG00099#2#CM087369.1,14095274,20715119,6619845,37,"{""active_hor(S2C18H1L)"": 4536062, ""cenSat(CER)...","{""CER"": 10815, ""HSAT5"": 6832, ""active"": 453606...",...,80141396.0,14095274,59426277.0,True,"[""active""]",True,1.0,True,True,chr18


In [2]:
print(censat_df.columns.tolist())


['sample_id', 'haplotype', 'assembly_id', 'sequence_id', 'region_start', 'region_end', 'region_size_bp', 'feature_count', 'region_label_bp_json', 'simplified_label_bp_json', 'contains_active_array', 'observed_large_array_count', 'observed_large_array_labels_json', 'has_multi_source_qc_flag', 'chrom_assignment', 'level', 'contig_size', 'dist_to_contig_start', 'dist_to_contig_end', 'region_not_near_contig_edge', 'reference_large_array_labels_json', 'all_large_arrays_represented', 'sequences_per_assembly_chrom', 'single_sequence_for_chrom', 'pass_qc', 'chrom']


In [ ]:
cat_index=/private/home/juklucas/github/censat_paper/data_tables/annotation/genes/cat_genes_hprc_r2_v1.3.index.csv

### Create bed files of censat regions for all samples 

In [1]:
import sys
from pathlib import Path
import pandas as pd
import glob

CENSAT_DIR   = '/private/groups/migalab/juklucas/censat_regions/censat_arrays/tables/censat_regions_pass_qc_by_chrom'
CAT_GENE_IDX = '/private/home/juklucas/github/censat_paper/data_tables/annotation/genes/cat_genes_hprc_r2_v1.3.index.csv'

outdir = Path(f'/private/groups/patenlab/mira/centrolign/analysis/Fig1_genes/extract_censat_from_gff3/all_chroms_out')
bed_dir = outdir / 'beds'
bed_dir.mkdir(parents=True, exist_ok=True)
# Load all chromosomes at once
all_tsvs = glob.glob(f'{CENSAT_DIR}/censat_regions_pass_qc_chr*.tsv')
censat_df = pd.concat(
    [pd.read_csv(f, sep='\t') for f in all_tsvs],
    ignore_index=True
)
censat_df['haplotype'] = censat_df['haplotype'].astype(str)
print(f'Loaded {len(censat_df)} censat regions across all chromosomes')

cat_idx = pd.read_csv(CAT_GENE_IDX)
cat_idx['haplotype'] = cat_idx['haplotype'].astype(str)

job_rows = []
for (sample_id, haplotype), regions in censat_df.groupby(['sample_id', 'haplotype']):
    sample_id = str(sample_id)
    haplotype = str(haplotype)

    match = cat_idx[
        (cat_idx['sample_id'] == sample_id) &
        (cat_idx['haplotype'] == haplotype)
    ]
    if match.empty:
        continue
    gff3_path = Path(match.iloc[0]['location'])
    if not gff3_path.exists():
        print(f'WARN: missing GFF3 for {sample_id} hap{haplotype}')
        continue

    # One BED per sample with all chromosomes' regions
    bed_path = bed_dir / f'{sample_id}.{haplotype}.bed'
    regions[['sequence_id', 'region_start', 'region_end', 'chrom_assignment']].to_csv(
        bed_path, sep='\t', header=False, index=False
    )


    job_rows.append({
        'sample_id': sample_id,
        'haplotype': haplotype,
        'gff3_path': str(gff3_path),
        'bed_path':  str(bed_path),
    })

jobs_path = outdir / 'jobs.tsv'
pd.DataFrame(job_rows).to_csv(jobs_path, sep='\t', index=False)
print(f'Wrote {len(job_rows)} jobs to {jobs_path}')
print(f'SLURM array range: 0-{len(job_rows) - 1}')

Loaded 4147 censat regions across all chromosomes
Wrote 462 jobs to /private/groups/patenlab/mira/centrolign/analysis/Fig1_genes/extract_censat_from_gff3/all_chroms_out/jobs.tsv
SLURM array range: 0-461
